In [8]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [15]:
##S3_BUCKET = "s3://ads508-housing-data-faye"
S3_BUCKET = "sagemaker-studio-mru71pwhjb"

PATHS = {
    "zhvi": f"s3://{S3_BUCKET}/Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv",
    "inventory": f"s3://{S3_BUCKET}/Metro_invt_fs_uc_sfrcondo_sm_month.csv",
    "market_temp": f"s3://{S3_BUCKET}/Metro_market_temp_index_uc_sfrcondo_month.csv",
    "sale_price": f"s3://{S3_BUCKET}/Metro_median_sale_price_uc_sfrcondo_sm_week.csv",
    "output": f"s3://{S3_BUCKET}/prepared/full_prepared_dataset.csv"
}

ID_COLS = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName"]
MERGE_KEYS = ["RegionID", "RegionName", "StateName", "Date"]

In [16]:
def basic_clean(df):
    df = df.drop_duplicates().copy()
    if "RegionType" in df.columns:
        df = df[df["RegionType"] == "msa"].copy()
    if "RegionName" in df.columns:
        df = df[df["RegionName"] != "United States"].copy()
    return df

def wide_to_long(df, value_name):
    date_cols = [c for c in df.columns if c not in ID_COLS]
    out = df.melt(
        id_vars=ID_COLS,
        value_vars=date_cols,
        var_name="Date",
        value_name=value_name
    )
    out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    out = out.dropna(subset=["Date"]).copy()
    return out

def load_and_prepare(path, value_name):
    df = pd.read_csv(path)
    df = basic_clean(df)
    df = wide_to_long(df, value_name)
    return df

def prepare_sale_price_monthly(path):
    df = pd.read_csv(path)
    df = basic_clean(df)
    df = wide_to_long(df, "MedianSalePrice")
    df["Date"] = df["Date"].dt.to_period("M").dt.to_timestamp("M")
    df = df.groupby(["RegionID", "RegionName", "StateName", "Date"], as_index=False)["MedianSalePrice"].mean()
    return df

def fill_within_region(df, cols):
    df = df.copy()
    for col in cols:
        df[col] = df.groupby("RegionName")[col].transform(lambda s: s.ffill().bfill())
    return df

def add_features(df):
    df = df.copy()
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["ZHVI_Lag1"] = df.groupby("RegionName")["ZHVI"].shift(1)
    df["Inventory_Lag1"] = df.groupby("RegionName")["Inventory"].shift(1)
    df["MarketTemp_Lag1"] = df.groupby("RegionName")["MarketTemp"].shift(1)
    df["ZHVI_PctChange"] = df.groupby("RegionName")["ZHVI"].pct_change()
    df["Inventory_PctChange"] = df.groupby("RegionName")["Inventory"].pct_change()
    return df

In [17]:
zhvi_long = load_and_prepare(PATHS["zhvi"], "ZHVI")
inventory_long = load_and_prepare(PATHS["inventory"], "Inventory")
market_temp_long = load_and_prepare(PATHS["market_temp"], "MarketTemp")
sale_price_monthly = prepare_sale_price_monthly(PATHS["sale_price"])

print(zhvi_long.shape)
print(inventory_long.shape)
print(market_temp_long.shape)
print(sale_price_monthly.shape)

(279822, 7)
(88065, 7)
(89919, 7)
(35424, 5)


In [18]:
df = (
    zhvi_long
    .merge(inventory_long[MERGE_KEYS + ["Inventory"]], on=MERGE_KEYS, how="left")
    .merge(market_temp_long[MERGE_KEYS + ["MarketTemp"]], on=MERGE_KEYS, how="left")
    .merge(sale_price_monthly, on=MERGE_KEYS, how="left")
)

print(df.shape)
df.head()

(279822, 10)


,RegionID,SizeRank,RegionName,RegionType,StateName,Date,ZHVI,Inventory,MarketTemp,MedianSalePrice
0,394913,1,"New York, NY",msa,NY,2000-01-31,216213.074441,NaN,NaN,NaN
1,753899,2,"Los Angeles, CA",msa,CA,2000-01-31,219357.633964,NaN,NaN,NaN
2,394463,3,"Chicago, IL",msa,IL,2000-01-31,149975.869412,NaN,NaN,NaN
3,394514,4,"Dallas, TX",msa,TX,2000-01-31,126453.868825,NaN,NaN,NaN
4,394692,5,"Houston, TX",msa,TX,2000-01-31,122700.097397,NaN,NaN,NaN


In [21]:
df = df.dropna(subset=["ZHVI"]).copy()
df = df.sort_values(["RegionName", "Date"]).copy()
df = fill_within_region(df, ["Inventory", "MarketTemp"])
df = df.dropna(subset=["Inventory", "MarketTemp"]).copy()
df = df.dropna(subset=["MedianSalePrice"]).copy()

print(df.isnull().sum())
print(df.shape)

RegionID               0
SizeRank               0
RegionName             0
RegionType             0
StateName              0
Date                   0
ZHVI                   0
Inventory              0
MarketTemp             0
MedianSalePrice        0
Year                   0
Month                  0
ZHVI_Lag1              0
Inventory_Lag1         0
MarketTemp_Lag1        0
ZHVI_PctChange         0
Inventory_PctChange    0
dtype: int64
(48622, 17)


In [23]:
df = add_features(df)
df = df.dropna(subset=["ZHVI_Lag1", "Inventory_Lag1", "MarketTemp_Lag1"]).copy()

print(df.shape)
df.head()

(48300, 17)


,RegionID,SizeRank,RegionName,RegionType,StateName,Date,ZHVI,Inventory,MarketTemp,MedianSalePrice,Year,Month,ZHVI_Lag1,Inventory_Lag1,MarketTemp_Lag1,ZHVI_PctChange,Inventory_PctChange
99476,394299,251,"Abilene, TX",msa,TX,2009-04-30,107899.903735,856.0,42.0,224318.333333,2009,4,107910.504329,856.0,42.0,-0.000098,0.0
100370,394299,251,"Abilene, TX",msa,TX,2009-05-31,107974.321882,856.0,42.0,224318.333333,2009,5,107899.903735,856.0,42.0,0.000690,0.0
101264,394299,251,"Abilene, TX",msa,TX,2009-06-30,107888.771914,856.0,42.0,224318.333333,2009,6,107974.321882,856.0,42.0,-0.000792,0.0
102158,394299,251,"Abilene, TX",msa,TX,2009-07-31,107865.541887,856.0,42.0,224318.333333,2009,7,107888.771914,856.0,42.0,-0.000215,0.0
103052,394299,251,"Abilene, TX",msa,TX,2009-08-31,107831.584511,856.0,42.0,224318.333333,2009,8,107865.541887,856.0,42.0,-0.000315,0.0


In [24]:
df.to_csv(PATHS["output"], index=False)
print("Saved to:", PATHS["output"])

Saved to: s3://sagemaker-studio-mru71pwhjb/prepared/full_prepared_dataset.csv
